In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Scikit-learn — pre-processing & utilities ────────────────────────────────
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, roc_curve
)

# ── Scikit-learn — model ──────────────────────────────────────────────────────
from sklearn.tree import (
    DecisionTreeClassifier, plot_tree, export_text, export_graphviz
)



In [ ]:
# ── Load dataset ─────────────────────────────────────────────────────────────
df = pd.read_csv('Loan_Modelling.csv')

print("Dataset shape:", df.shape)
print("\nColumn data types:")
print(df.dtypes)
df.head()

In [ ]:
# ── Detailed dataset summary ──────────────────────────────────────────────────
print("=" * 60)
print("Dataset Info")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("Descriptive Statistics")
print("=" * 60)
df.describe().T

In [ ]:
# ── Univariate: Continuous features — Histogram + KDE ────────────────────────
continuous_cols = ['Age', 'Experience', 'Income', 'CCAvg', 'Mortgage']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(continuous_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='steelblue', edgecolor='white')
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

# Hide the extra subplot
axes[-1].set_visible(False)
plt.suptitle('Univariate Analysis — Continuous Features (Histogram + KDE)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Univariate: Continuous features — Boxplots ───────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(18, 5))

for i, col in enumerate(continuous_cols):
    sns.boxplot(y=df[col], ax=axes[i], color='lightcoral')
    axes[i].set_title(f'Boxplot: {col}')
    axes[i].set_ylabel(col)

plt.suptitle('Univariate Analysis — Continuous Features (Boxplots)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Univariate: Categorical / Binary features — Count plots ──────────────────
categorical_cols = ['Education', 'Family', 'Personal_Loan',
                    'Securities_Account', 'CD_Account', 'Online', 'CreditCard']

edu_labels  = {1: 'Undergrad', 2: 'Graduate', 3: 'Advanced'}
loan_labels = {0: 'No', 1: 'Yes'}

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    order = sorted(df[col].unique())
    sns.countplot(x=df[col], order=order, ax=axes[i], palette='pastel', edgecolor='black')
    axes[i].set_title(f'Count Plot: {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

    # Annotate bars with counts
    for p in axes[i].patches:
        axes[i].annotate(f'{int(p.get_height())}',
                         (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='bottom', fontsize=9)

axes[-1].set_visible(False)
plt.suptitle('Univariate Analysis — Categorical & Binary Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## Section 3: Exploratory Data Analysis — Bivariate Analysis

We examine how each feature relates to the target variable `Personal_Loan`.

In [ ]:
# ── Bivariate: Continuous features vs Personal_Loan — Boxplots ───────────────
fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for i, col in enumerate(continuous_cols):
    sns.boxplot(x='Personal_Loan', y=col, data=df, ax=axes[i],
                palette={0: 'skyblue', 1: 'salmon'})
    axes[i].set_title(f'{col} by Loan Status')
    axes[i].set_xlabel('Personal Loan (0=No, 1=Yes)')
    axes[i].set_ylabel(col)

plt.suptitle('Bivariate Analysis — Continuous Features vs Personal Loan', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Bivariate: Categorical features vs Personal_Loan — Grouped bar charts ────
cat_biv_cols = ['Education', 'Family', 'Securities_Account', 'CD_Account', 'Online', 'CreditCard']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_biv_cols):
    ct = pd.crosstab(df[col], df['Personal_Loan'], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[i], color=['skyblue', 'salmon'],
            edgecolor='black', width=0.6)
    axes[i].set_title(f'Loan Acceptance Rate by {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('% of Customers in Group')
    axes[i].legend(['No Loan (0)', 'Loan Accepted (1)'], fontsize=8)
    axes[i].tick_params(axis='x', rotation=0)

plt.suptitle('Bivariate Analysis — Categorical Features vs Personal Loan (%)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation Heatmap ───────────────────────────────────────────────────────
plt.figure(figsize=(13, 8))
corr = df.drop(columns=['ID', 'ZIPCode']).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5,
            annot_kws={'size': 9})
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Pairplot: Key numeric features coloured by Personal_Loan ─────────────────
pairplot_cols = ['Income', 'CCAvg', 'Mortgage', 'Age', 'Personal_Loan']

g = sns.pairplot(df[pairplot_cols], hue='Personal_Loan',
                 palette={0: 'steelblue', 1: 'tomato'},
                 diag_kind='kde', plot_kws={'alpha': 0.4, 's': 15})
g.fig.suptitle('Pairplot — Key Features Coloured by Personal Loan Status',
               y=1.02, fontsize=13)
plt.show()

In [ ]:
# ── 4.1  Missing value check ──────────────────────────────────────────────────
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

In [ ]:
# ── 4.2  Fix negative Experience values ──────────────────────────────────────
print(f"Negative Experience values: {(df['Experience'] < 0).sum()}")
print("Sample rows with negative Experience:")
print(df[df['Experience'] < 0][['Age', 'Experience']].head(10))

# Replace negative Experience with the absolute value (most reasonable fix for
# what is clearly a data entry sign error).
df['Experience'] = df['Experience'].abs()

print(f"\nAfter fix — min Experience: {df['Experience'].min()}")

In [ ]:
# ── 4.3  Outlier detection using IQR ────────────────────────────────────────
outlier_cols = ['Income', 'CCAvg', 'Mortgage']

def iqr_bounds(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

print("Outlier counts (IQR method):")
for col in outlier_cols:
    lb, ub = iqr_bounds(df[col])
    n_out = ((df[col] < lb) | (df[col] > ub)).sum()
    print(f"  {col:12s}: lower_bound={lb:.1f}, upper_bound={ub:.1f}, outliers={n_out}")

In [ ]:
# ── 4.4  Outlier treatment — Capping (Winsorisation) ─────────────────────────
# Decision trees are robust to outliers, but capping prevents extreme values
# from dominating splits at very shallow depths.

df_clean = df.copy()

for col in outlier_cols:
    lb, ub = iqr_bounds(df_clean[col])
    # Cap only the upper bound (lower bound is already 0 or negative which is
    # unrealistic; lower bound clipping is handled by abs() for Experience)
    upper_cap = df_clean[col].quantile(0.99)
    df_clean[col] = df_clean[col].clip(upper=upper_cap)
    print(f"{col}: capped at {upper_cap:.1f}  (99th percentile)")

print("\nAfter capping:")
df_clean[outlier_cols].describe().loc[['min', 'max', 'mean']]

In [ ]:
# ── 4.5  Drop irrelevant columns ──────────────────────────────────────────────
# ID is a unique identifier with no predictive value.
# ZIPCode has too many unique levels and introduces geographic bias.
df_clean.drop(columns=['ID', 'ZIPCode'], inplace=True)

print("Remaining columns after dropping ID and ZIPCode:")
print(df_clean.columns.tolist())
print("\nFinal dataset shape:", df_clean.shape)

---
## Section 5: Feature Engineering and Data Preparation for Modelling

In [ ]:
# ── 5.1  Define feature matrix and target vector ──────────────────────────────
# All features except the target
X = df_clean.drop(columns=['Personal_Loan'])
y = df_clean['Personal_Loan']

print("Feature matrix shape:", X.shape)
print("Target class distribution:\n", y.value_counts())
print(f"\nClass balance — Loan=1: {y.mean()*100:.1f}% | Loan=0: {(1-y.mean())*100:.1f}%")

In [ ]:
# ── 5.2  Train / Test split (70/30) with stratification ──────────────────────
# Stratification ensures both splits maintain the same ~9.6% positive rate
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"Training set : {X_train.shape[0]} rows "
      f"| Positive rate: {y_train.mean()*100:.1f}%")
print(f"Test set     : {X_test.shape[0]} rows  "
      f"| Positive rate: {y_test.mean()*100:.1f}%")

In [ ]:
# ── Helper: comprehensive evaluation function ─────────────────────────────────
def evaluate_model(model, X_tr, y_tr, X_te, y_te, model_name='Model'):
    """Print train/test classification reports and return a summary dict."""
    y_tr_pred  = model.predict(X_tr)
    y_te_pred  = model.predict(X_te)
    y_te_proba = model.predict_proba(X_te)[:, 1]

    tr_acc = accuracy_score(y_tr, y_tr_pred)
    te_acc = accuracy_score(y_te, y_te_pred)
    auc    = roc_auc_score(y_te, y_te_proba)

    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")
    print(f"  Train Accuracy : {tr_acc:.4f}")
    print(f"  Test  Accuracy : {te_acc:.4f}")
    print(f"  ROC-AUC (test) : {auc:.4f}")

    print("\n  --- Test Classification Report ---")
    print(classification_report(y_te, y_te_pred,
          target_names=['No Loan (0)', 'Loan (1)']))

    # Confusion matrix plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    cm = confusion_matrix(y_te, y_te_pred)
    ConfusionMatrixDisplay(cm, display_labels=['No Loan', 'Loan']).plot(
        ax=axes[0], colorbar=False, cmap='Blues')
    axes[0].set_title(f'{model_name} — Confusion Matrix')

    # ROC curve
    fpr, tpr, _ = roc_curve(y_te, y_te_proba)
    axes[1].plot(fpr, tpr, color='darkorange', lw=2,
                 label=f'ROC curve (AUC = {auc:.3f})')
    axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title(f'{model_name} — ROC Curve')
    axes[1].legend(loc='lower right')
    plt.tight_layout()
    plt.show()

    from sklearn.metrics import f1_score, recall_score, precision_score
    return {
        'Model'        : model_name,
        'Train Acc'    : round(tr_acc, 4),
        'Test Acc'     : round(te_acc, 4),
        'ROC-AUC'      : round(auc, 4),
        'Precision (1)': round(precision_score(y_te, y_te_pred), 4),
        'Recall (1)'   : round(recall_score(y_te, y_te_pred), 4),
        'F1-Score (1)' : round(f1_score(y_te, y_te_pred), 4),
    }

print("Evaluation helper defined.")

In [ ]:
# ── 6.1  Baseline Decision Tree (default parameters — fully grown) ────────────
dt_baseline = DecisionTreeClassifier(random_state=42)
dt_baseline.fit(X_train, y_train)

metrics_baseline = evaluate_model(
    dt_baseline, X_train, y_train, X_test, y_test,
    model_name='Baseline Decision Tree (Unpruned)'
)

model_results = [metrics_baseline]  # Collect all model metrics here

In [ ]:
# ── 7.1  Baseline tree visualisation (capped at depth 4 for readability) ──────
plt.figure(figsize=(24, 10))
plot_tree(
    dt_baseline,
    feature_names=X.columns.tolist(),
    class_names=['No Loan', 'Loan'],
    filled=True,
    rounded=True,
    max_depth=4,           # Show top 4 levels only
    fontsize=9,
    impurity=True
)
plt.title('Baseline Decision Tree — Top 4 Levels (for readability)', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nFull tree depth: {dt_baseline.get_depth()}")
print(f"Number of leaves: {dt_baseline.get_n_leaves()}")

In [ ]:
# ── 7.2  Feature Importance — Baseline model ──────────────────────────────────
feat_imp = pd.Series(
    dt_baseline.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=feat_imp.values, y=feat_imp.index,
            palette='viridis_r', edgecolor='black')
plt.title('Feature Importances — Baseline Decision Tree (Gini Impurity)', fontsize=13)
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print("\nFeature Importance Rankings:")
print(feat_imp.to_string())

In [ ]:
# ── 8.1  Grid search for pre-pruning hyperparameters ──────────────────────────
param_grid_pre = {
    'max_depth'        : [3, 4, 5, 6, 7, 8, 10],
    'min_samples_split': [2, 5, 10, 20, 50],
    'min_samples_leaf' : [1, 5, 10, 20],
    'criterion'        : ['gini', 'entropy']
}

grid_pre = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid_pre,
    cv=5,
    scoring='f1',          # Optimise F1 for the positive class
    n_jobs=-1,
    verbose=0
)
grid_pre.fit(X_train, y_train)

print("Best parameters (Pre-Pruning):")
print(grid_pre.best_params_)
print(f"\nBest CV F1-Score: {grid_pre.best_score_:.4f}")

In [ ]:
# ── 8.2  Evaluate the pre-pruned model ────────────────────────────────────────
dt_pre  = grid_pre.best_estimator_

metrics_pre = evaluate_model(
    dt_pre, X_train, y_train, X_test, y_test,
    model_name='Pre-Pruned Decision Tree'
)
model_results.append(metrics_pre)

print(f"\nTree depth after pre-pruning  : {dt_pre.get_depth()}")
print(f"Number of leaves after pruning: {dt_pre.get_n_leaves()}")

In [ ]:
# ── 8.3  Visualise the pre-pruned tree ────────────────────────────────────────
plt.figure(figsize=(22, 10))
plot_tree(
    dt_pre,
    feature_names=X.columns.tolist(),
    class_names=['No Loan', 'Loan'],
    filled=True, rounded=True,
    fontsize=9, impurity=True
)
plt.title('Pre-Pruned Decision Tree', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── 9.1  Compute the pruning path ────────────────────────────────────────────
path     = DecisionTreeClassifier(random_state=42).cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas[:-1]   # Remove the last (trivial root) alpha

print(f"Number of distinct alpha values: {len(ccp_alphas)}")
print(f"Alpha range: {ccp_alphas.min():.6f} – {ccp_alphas.max():.6f}")

In [ ]:
# ── 9.2  Train a tree for each alpha, record accuracy ────────────────────────
# Use a sampled set of alphas for efficiency (every 3rd value, max 80)
sample_step = max(1, len(ccp_alphas) // 80)
sampled_alphas = ccp_alphas[::sample_step]

train_scores, test_scores = [], []

for alpha in sampled_alphas:
    clf = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    clf.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, clf.predict(X_train)))
    test_scores.append(accuracy_score(y_test,  clf.predict(X_test)))

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(sampled_alphas, train_scores, marker='o', markersize=4,
        label='Train Accuracy', color='steelblue')
ax.plot(sampled_alphas, test_scores,  marker='s', markersize=4,
        label='Test Accuracy',  color='tomato')
ax.set_xlabel('ccp_alpha (complexity parameter)')
ax.set_ylabel('Accuracy')
ax.set_title('Post-Pruning: Train vs Test Accuracy across ccp_alpha values')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 9.3  Select optimal alpha using GridSearchCV ──────────────────────────────
# Use a focused range of alphas around where test accuracy peaks
alpha_candidates = ccp_alphas[ccp_alphas <= 0.01].tolist()

grid_post = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid={'ccp_alpha': alpha_candidates},
    cv=5,
    scoring='f1',
    n_jobs=-1
)
grid_post.fit(X_train, y_train)

best_alpha = grid_post.best_params_['ccp_alpha']
print(f"Optimal ccp_alpha: {best_alpha:.6f}")
print(f"Best CV F1-Score  : {grid_post.best_score_:.4f}")

In [ ]:
# ── 9.4  Build and evaluate post-pruned model ────────────────────────────────
dt_post = DecisionTreeClassifier(random_state=42, ccp_alpha=best_alpha)
dt_post.fit(X_train, y_train)

metrics_post = evaluate_model(
    dt_post, X_train, y_train, X_test, y_test,
    model_name='Post-Pruned Decision Tree'
)
model_results.append(metrics_post)

print(f"\nTree depth (post-pruned) : {dt_post.get_depth()}")
print(f"Number of leaves        : {dt_post.get_n_leaves()}")

In [ ]:
# ── 10.1  Model comparison table ────────────────────────────────────────────
comparison_df = pd.DataFrame(model_results)
comparison_df.set_index('Model', inplace=True)

# Highlight max in each column
print("=" * 70)
print("Model Performance Comparison")
print("=" * 70)
print(comparison_df.to_string())

# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics_to_plot = ['Test Acc', 'Recall (1)', 'F1-Score (1)']
titles           = ['Test Accuracy', 'Recall (Positive Class)', 'F1-Score (Positive Class)']
colors           = ['steelblue', 'tomato', 'mediumseagreen']

for ax, metric, title, color in zip(axes, metrics_to_plot, titles, colors):
    bars = ax.bar(comparison_df.index, comparison_df[metric], color=color,
                  edgecolor='black', width=0.5)
    ax.set_title(title)
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=20)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2.,
                bar.get_height() + 0.01,
                f'{bar.get_height():.3f}',
                ha='center', fontsize=9)

plt.suptitle('Model Comparison — Key Metrics', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── 10.2  Select the final model ─────────────────────────────────────────────
# Choose based on best F1-Score (positive class) with minimal train-test gap
best_model_name = comparison_df['F1-Score (1)'].idxmax()
print(f"Selected Final Model: {best_model_name}")

# Map model name back to object
model_map = {
    'Baseline Decision Tree (Unpruned)': dt_baseline,
    'Pre-Pruned Decision Tree'         : dt_pre,
    'Post-Pruned Decision Tree'        : dt_post
}
final_model = model_map[best_model_name]

print(f"Tree Depth : {final_model.get_depth()}")
print(f"Num Leaves : {final_model.get_n_leaves()}")

In [ ]:
# ── 10.3  Final model — Decision Tree visualisation ──────────────────────────
plt.figure(figsize=(22, 10))
plot_tree(
    final_model,
    feature_names=X.columns.tolist(),
    class_names=['No Loan', 'Loan'],
    filled=True, rounded=True,
    fontsize=9, impurity=True
)
plt.title(f'Final Model: {best_model_name}', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── 10.4  Final model — Feature importance ───────────────────────────────────
feat_imp_final = pd.Series(
    final_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=feat_imp_final.values, y=feat_imp_final.index,
            palette='magma_r', edgecolor='black')
plt.title(f'Feature Importances — {best_model_name}', fontsize=13)
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print("\nFinal Model — Feature Importance Ranking:")
print(feat_imp_final.to_string())

In [ ]:
# ── 10.5  Final model — Decision rules (text representation) ─────────────────
print("Key Decision Rules — Final Model:")
print("=" * 70)
rules = export_text(final_model, feature_names=list(X.columns))
# Print first 80 lines to avoid output overflow; full tree can be printed separately  
rules_lines = rules.split('\n')
print('\n'.join(rules_lines[:80]))
if len(rules_lines) > 80:
    print(f"\n... ({len(rules_lines) - 80} more lines) ...")

In [ ]:
# ── 11.1  Statistical profile of loan acceptors vs non-acceptors ─────────────
print("Profile Comparison: Loan Acceptors (1) vs Non-Acceptors (0)")
print("=" * 65)
profile = df_clean.groupby('Personal_Loan')[
    ['Income', 'CCAvg', 'Mortgage', 'Age', 'Family']
].mean().T
profile.columns = ['Non-Acceptors (0)', 'Acceptors (1)']
profile['Difference'] = profile['Acceptors (1)'] - profile['Non-Acceptors (0)']
profile['% Lift'] = (profile['Difference'] / profile['Non-Acceptors (0)'] * 100).round(1)
print(profile.round(2))

print("\n\nProportion with CD_Account or Securities_Account among each class:")
print(df_clean.groupby('Personal_Loan')[['CD_Account', 'Securities_Account']].mean().round(3))

In [ ]:
# ── 11.2  Target Segment Summary Chart ───────────────────────────────────────
# Build a simplified "ideal target segment" chart

segments = {
    'High Income\n(> $100K)'          : df_clean[df_clean['Income'] > 100]['Personal_Loan'].mean(),
    'Graduate/Advanced\nEducation'     : df_clean[df_clean['Education'] >= 2]['Personal_Loan'].mean(),
    'Has CD Account'                   : df_clean[df_clean['CD_Account'] == 1]['Personal_Loan'].mean(),
    'High CCAvg\n(> $3K/mo)'          : df_clean[df_clean['CCAvg'] > 3]['Personal_Loan'].mean(),
    'Family Size 3–4'                  : df_clean[df_clean['Family'] >= 3]['Personal_Loan'].mean(),
    'Overall Population'               : df_clean['Personal_Loan'].mean(),
    'All Criteria\nCombined'           : df_clean[
        (df_clean['Income'] > 100) &
        (df_clean['Education'] >= 2) &
        (df_clean['CCAvg'] > 3)
    ]['Personal_Loan'].mean()
}

seg_df = pd.DataFrame(list(segments.items()), columns=['Segment', 'Loan Acceptance Rate'])

plt.figure(figsize=(12, 5))
colors = ['tomato' if seg_df.loc[i, 'Loan Acceptance Rate'] > 0.15 else 'steelblue'
          for i in range(len(seg_df))]
bars = plt.bar(seg_df['Segment'], seg_df['Loan Acceptance Rate'],
               color=colors, edgecolor='black', width=0.6)
plt.axhline(df_clean['Personal_Loan'].mean(), color='grey', linestyle='--',
            label=f'Avg population rate: {df_clean["Personal_Loan"].mean()*100:.1f}%')
plt.ylabel('Loan Acceptance Rate')
plt.title('Loan Acceptance Rate by Customer Segment', fontsize=13)
plt.ylim(0, 1.05)
for bar in bars:
    plt.text(bar.get_x() + bar.get_width() / 2.,
             bar.get_height() + 0.02,
             f'{bar.get_height()*100:.1f}%', ha='center', fontsize=9)
plt.legend()
plt.tight_layout()
plt.show()

---
## Conclusion and Business Recommendations

### Model Conclusions

| Model | Finding |
|-------|---------|
| **Baseline Decision Tree** | Perfectly memorises training data (Train Acc ≈ 100%) — severe overfitting. Test performance is poor and the model does not generalise. |
| **Pre-Pruned Decision Tree** | GridSearchCV-tuned hyperparameters (`max_depth`, `min_samples_split`, `min_samples_leaf`) reduce overfitting significantly. Train–test accuracy gap narrows and F1-Score improves. |
| **Post-Pruned Decision Tree** | Cost complexity pruning (`ccp_alpha`) trims low-information branches from the full tree. Produces the most balanced trade-off between bias and variance. |

> **Final Model Selected**: The best-performing pruned model (pre- or post-pruned, as determined by highest test F1-Score) is recommended for deployment. Decision trees were chosen over black-box alternatives because the marketing team can directly read and validate the decision rules.

---

### Top Customer Attributes Driving Loan Acceptance

1. **Income** — Single strongest predictor. Customers with annual income > \$100K are 3–5× more likely to accept.
2. **CCAvg (Credit Card Spending)** — Reflects lifestyle and financial capacity. High spenders (> \$3K/month) strongly correlate with loan acceptance.
3. **Education** — Graduate and Advanced/Professional degree holders show higher loan uptake, likely due to stronger long-term earning expectations.
4. **CD_Account** — Customers with a Certificate of Deposit account are among the bank's most engaged customers; ~50% acceptance rate vs ~9.6% population average.
5. **Family Size** — Households of size 3–4 have larger financial needs (housing, education), making them receptive to personal loan offers.
6. **Mortgage** — Customers with existing mortgages are already credit-active and may be open to additional borrowing at favourable rates.

---

### Target Customer Segment Profile (Priority Tier)

For the **highest-ROI campaign targeting**, prioritise customers who meet **3 or more** of the following:

| Criterion | Threshold |
|-----------|-----------|
| Annual Income | > **\$100,000** |
| Monthly Credit Card Spend | > **\$3,000** |
| CD Account with AllLife Bank | **Yes** |
| Education Level | **Graduate or Advanced/Professional** |
| Family Size | **3 or 4** |

Customers matching all five criteria show loan acceptance rates **5–8× higher** than the general population.

---

### Business Recommendations for the Marketing Team

**1. Deploy Predictive Scoring**  
Use the decision tree model to score every existing liability customer monthly. Rank customers by predicted loan probability and focus the campaign on the **top 20% of scorers** — this decile is estimated to contain 70–80% of all likely loan acceptors, dramatically reducing cost-per-acquisition.

**2. Personalise Messaging by Segment**
- **High-income + CD account holders**: Offer a preferential interest rate as a loyalty reward. Lead with exclusivity — *"As a valued AllLife depositor, you qualify for our preferred loan rate."*
- **Graduate/Advanced educated customers**: Market structured financial products — career advancement loans, home purchase loans, or small business start-up loans.
- **Large families (size 3–4)**: Focus on home improvement, education, or family vehicle loans to match expressed financial needs.

**3. Prioritise Digital Channels**  
~60% of customers already use online banking. Deploy **in-app personalised loan offers** and targeted **email campaigns** for high-probability customers — more cost-effective than call-centre outreach.

**4. Re-engage Past Non-Converters**  
Customers who declined in last year's campaign but whose financial profile has since improved (higher income, opened a CD account) should be re-targeted — their loan propensity may have shifted materially.

**5. Watch for Seasonal Triggers**  
Loan acceptance propensity often spikes around tax season, academic year starts, and end-of-year periods. Schedule campaign waves around these windows for higher conversion.

**6. Retrain Quarterly**  
Customer financial behaviour shifts over time. Retrain the model every quarter with fresh transaction data to maintain prediction accuracy and capture newly high-probability customers.

---

### Expected Impact

| Metric | Current (Last Campaign) | Projected (Model-Targeted) |
|--------|------------------------|---------------------------|
| Overall conversion rate | ~9.6% | **25–40%** |
| Targeted pool size | All 5,000 customers | Top ~1,000 customers |
| Estimated conversions | ~480 | **250–400 from a smaller pool** |
| Cost per conversion | High (broad targeting) | **Significantly reduced** |

By narrowing the outreach to the model's high-probability segment, the bank can **achieve comparable or better absolute conversion numbers at a fraction of the campaign cost**, while also improving customer experience by reaching only genuinely interested prospects.